# Phi 4 Model

In [ ]:
"""! @brief Simple/Complex Text Classification with Phi-4 Mini LLM"""
# @file Phi4_LLM.ipynb
#
# @mainpage Phi-4 Mini
#
# @section description_main Description
# This project evaluates and compares the simple/complex text classification Phi-4 with and without few-shots.
# The idea is to compare the performance of the configurations in terms of accuracy
#
# @section libraries_main Libraries
# - gc (https://docs.python.org/3/library/gc.html)
#   - Used to release unused GPU memory
#
# @section imports_main Imports
# - LLMs (https://github.com/Luco1421/TPs_IA/blob/master/TP2/LLMs.py)
#   - Used to read the dataset and some test LLM's functions
#
# @section models_main Models
# - Phi-4 Mini (https://huggingface.co/microsoft/Phi-4-mini-instruct)
#   - Model to evaluate
#
# @section authors_main Authors
# - Alejandro Cerdas
# - Kener Castillo
# - Pablo Perez

# Imports

In [ ]:
# @brief Initialization of dataset and LLM utility objects.

# Import
import gc

# Import all utilities and classes from the LLMs module
from LLMs import *

# Load dataset from Excel file
dataset = ReadDataset("FEINA_1.xlsx").read()

# Create utility instance for LLM testing
llm_utils = TestLLMUtils()

## Charge model


In [ ]:
# @brief GPU cleanup and SmolLM3 model initialization.

# Release unused GPU memory resources
clean_gpu()

# Load pretrained SmolLM3 language model
phi_4 = charge_model("microsoft/Phi-4-mini-instruct")

## Function to request Phi 4's answer


In [ ]:
def chat_phi_4(text: str, prompt: str):
    """! Generates a response using the Phi-4 model.

    @details
    This function formats a conversation using the chat
    template of the tokenizer, tokenizes the input,
    performs text generation with the language model,
    decodes the generated response, and extracts
    the final processed result.

    @param text User input text.
    @param prompt System prompt used to guide generation.

    @return Processed generated response.
    """
    # Define conversation messages
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": text},
    ]

    # Apply chat template and tokenize input
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # Disable gradient computation during inference
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            top_p=0.1,
            do_sample=True
        )
    # Decode generated response tokens
    response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    # Release GPU memory
    clean_gpu()
    # Process generated response to extract labels
    return extract_results(response_text)

# Extract tokenizer and model from the loaded Phi-4 mini tuple
tokenizer, model = phi_4

### Example of use without shots


In [ ]:
# @Brief Example of use of Phi 4

# Generate response using predefined prompt
chat_phi_4(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + TEXT_BEGINNER)

### Example of use with shots

In [ ]:
# @brief Executes few-shot inference experiments with Phi-4-mini.

# Create batching utility instance
batcher = Batcher()

# Evaluate the model with different shot configurations
for i in [2,4,7]:

    # Generate few-shot examples from the dataset
    shot = batcher.set_shots(dataset.corpus[-i:],dataset.labels[-i:])

    # Generate response using few-shot prompting
    chat_phi_4(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + SHOTS_BEGINNER + shot + TEXT_BEGINNER)

# Results

# Test without Few Shots

In [ ]:
# @brief Test model without Few Shots

# Evaluate model performance without few-shots using all data
llm_utils.test_without_shots("Phi 4", chat_phi_4, dataset)

# Test with Few Shots

In [ ]:
# @brief Test model with Few Shots with all data

# Evaluate model performance with few-shots using different configurations of shots
llm_utils.test_with_shots("Phi 4", chat_phi_4, dataset, [2, 4, 7])

# Test with all data

In [ ]:
# @brief Evaluates SmolLM3 using 30 different random partitions (splits) of the dataset

# Run full evaluation for SmolLM3
llm_utils.test_all_LLM("Phi 4", chat_phi_4, dataset, [2])

In [ ]:
# @brief Releases GPU memory and clears model-related variables.

# Clean GPU cached memory
clean_gpu()

# Delete model, tokenizer, and inference function references
del phi_4, chat_phi_4, tokenizer, model

# Force Python garbage collector to release memory
gc.collect()